# Data Analysis Pipeline

This notebook builds a complete, multi‑step data analysis pipeline that combines **pandas**, **NumPy**, and **matplotlib** into a single cohesive tool.

1. Generate or locate raw data
2. Load the data
3. Clean missing values while recording their original counts
4. Compute summary statistics
5. Visualize distributions
6. Export a text report

Each step is an independent function, making the pipeline modular and reusable.

## Environment Setup

The three core libraries are imported, along with `os` for directory operations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

---
## 1. Create the sample dataset

The same dictionary data used in previous projects is converted to a CSV file. In a real pipeline, this step would be replaced by pointing to an existing data source.

In [ ]:
data = {
    "age": [25, 30, 35, None, 40, 120, 29, 31],
    "income": [50000, 60000, 75000, 80000, None, 95000, 62000, 58000],
    "score": [85, 90, 78, 92, 88, None, 76, 95]
}

def convert_csv(data):
    """Write the dictionary to a CSV file in the current directory."""
    pd.DataFrame(data).to_csv('sample_data.csv', index=False)

---
## 2. Load the data

`load_data` reads the CSV back into a pandas DataFrame.

In [ ]:
def load_data(filepath):
    """Read a CSV file and return a DataFrame."""
    return pd.read_csv(filepath)

---
## 3. Clean the data

`clean_data` does three things:
1. Creates a **copy** of the input DataFrame so the original remains untouched.
2. Counts how many missing values each numeric column originally had.
3. Fills those missing values with the column **median**.

The median was chosen over the mean because it is robust to outliers. For the `age` column, the 120‑year‑old outlier inflates the mean to ~44, while the median stays at 31. Filling with the median produces a more realistic replacement.

The function returns the filled DataFrame and the missing‑count dictionary separately so the counts can be passed to the analysis step.

In [ ]:
def clean_data(raw_df):
    """Copy the DataFrame, record missing counts, fill numeric columns with median."""
    df = raw_df.copy()
    missing_counts = {}
    for col in df.select_dtypes(include="number").columns:
        missing_counts[col] = df[col].isnull().sum()
        df[col] = df[col].fillna(df[col].median())
    return df, missing_counts

---
## 4. Analyze the data

`analyze_data` converts each numeric column into a NumPy array and computes summary statistics using NaN‑aware functions. Although the data was already cleaned, the NaN‑aware functions are kept as a defensive habit.

The `missing_count` value comes from the dictionary captured in `clean_data`.

### Functions used
- `np.nanmean()` - mean, skipping NaN
- `np.nanmedian()` - median, skipping NaN
- `np.nanstd()` - standard deviation, skipping NaN
- `np.nanmin()` and `np.nanmax()` - extremes (lowest and highest values), skipping NaN
- `np.isnan().sum()` - count of NaN entries (the pre‑cleaned count is used instead)

In [ ]:
def analyze_data(df, missing_values):
    """Return a nested dictionary of statistics for each numeric column."""
    summary = {}
    for col in df.select_dtypes(include="number").columns:
        np_data = {}
        np_list = np.array(df[col], dtype=float)
        np_data["mean"] = np.nanmean(np_list)
        np_data["median"] = np.nanmedian(np_list)
        np_data["std"] = np.nanstd(np_list)
        np_data["min"] = np.nanmin(np_list)
        np_data["max"] = np.nanmax(np_list)
        np_data["missing_values"] = missing_values[col]
        summary[col] = np_data
    return summary

---
## 5. Plot distributions (with directory management)

`plot_distribution` creates a histogram for each numeric column and saves it as a PNG file inside a `histograms` subdirectory. 

The function ensures the `histograms` folder exists before saving. After each plot, `plt.close()` clears the figure so that bars and labels from one plot would not get carried into the next. Alternatively, `plt.clf()` would reset the same figure for reuse, but `plt.close()` is safer when saving many files sequentially.

In [ ]:
def plot_distribution(df):
    """Save one histogram per numeric column inside a 'histograms' folder."""
    os.makedirs("histograms", exist_ok=True)        # create folder if missing
    for col in df.select_dtypes(include="number").columns:
        plt.hist(df[col], 4)
        plt.title(f"{col} distribution")
        plt.xlabel(col)
        plt.ylabel("frequency")
        filepath = os.path.join("histograms", f"{col}.png")
        plt.savefig(filepath)
        plt.close()   # prevent carry‑over between plots; plt.clf() would reuse the same figure

---
## 6. Export a text report

`export_report` writes a `report.txt` file that contains:
- A summary of the dataset dimensions.
- The full statistics dictionary for each column.
- A list of the saved histogram filenames.

In [ ]:
def export_report(summary, df):
    """Write a data analysis report to report.txt."""
    report_lines = []
    report_lines.append("Data Analysis Report")
    report_lines.append(f"Rows: {len(df)}, Columns: {len(df.columns)}")
    for col in summary:
        report_lines.append(f"{col} = {summary[col]}")
    report_lines.append("Histograms:")
    for col in df.columns:
        report_lines.append(f"{col}.png")
    with open("report.txt", "w") as f:
        f.write("\n".join(report_lines))

---
## 7. Run the full pipeline

All the functions are called in sequence. Each step passes its output to the next. The `raw_data` DataFrame remains untouched throughout, because `clean_data` operates on a copy.

In [ ]:
convert_csv(data)
raw_data = load_data("sample_data.csv")
cleaned_df, missing_counts = clean_data(raw_data)
plot_distribution(cleaned_df)
export_report(analyze_data(cleaned_df, missing_counts), cleaned_df)

---
This pipeline represents the standard "load, clean, analyze, visualize, report" workflow.